# Uninformed search (BFS, DFS, UCS) — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
DIRS=[(1,0),(-1,0),(0,1),(0,-1)]
def neighbors(p, shape=(5,5), walls=set()):
    for dr,dc in DIRS:
        q=(p[0]+dr,p[1]+dc)
        if 0<=q[0]<shape[0] and 0<=q[1]<shape[1] and q not in walls: yield q
def reconstruct(parent, goal):
    path=[]; x=goal
    while x is not None: path.append(x); x=parent.get(x)
    return path[::-1]
def draw(vals, title):
    plt.figure(figsize=(5,3)); plt.plot(vals,'-o'); plt.title(title)
def draw_grid(path=(), explored=(), frontier=(), walls=set(), start=(0,0), goal=(4,4), shape=(5,5), title='grid'):
    img=np.zeros(shape)
    for r,c in walls: img[r,c]=-1
    for r,c in explored: img[r,c]=.35
    for r,c in frontier: img[r,c]=.65
    for r,c in path: img[r,c]=1
    plt.figure(figsize=(4,4)); plt.imshow(img,cmap='viridis',vmin=-1,vmax=1); plt.scatter([start[1]],[start[0]],c='w',edgecolor='k'); plt.scatter([goal[1]],[goal[0]],c='r',edgecolor='k'); plt.title(title); plt.xticks(range(shape[1])); plt.yticks(range(shape[0])); plt.grid(color='w',alpha=.3)
print('setup ok')

## Frontier rules without heuristic knowledge
BFS, DFS, and UCS differ only in how they order the frontier; the plots make that ordering visible.

In [ ]:
from collections import deque
s=(0,0); g=(4,4); q=deque([s]); par={s:None}; ex=[]
while q:
 u=q.popleft(); ex.append(u)
 if u==g: break
 for v in neighbors(u):
  if v not in par: par[v]=u; q.append(v)
p=reconstruct(par,g); draw_grid(p,ex,q,title=f'BFS length {len(p)-1}')
assert len(p)-1==8 and ex[-1]==g

In [ ]:
s=(0,0); g=(4,4); st=[s]; par={s:None}; ex=[]
while st:
 u=st.pop()
 if u in ex: continue
 ex.append(u)
 if u==g: break
 for v in neighbors(u):
  if v not in par: par[v]=u; st.append(v)
p=reconstruct(par,g); draw_grid(p,ex,st,title=f'DFS length {len(p)-1}')
assert len(p)-1==16 and p[-1]==g

In [ ]:
import heapq
s=(0,0); g=(4,4); cost={(2,2):9,(2,3):9}; h=[(0,s)]; d={s:0}; par={s:None}; ex=[]
while h:
 du,u=heapq.heappop(h)
 if du!=d[u]: continue
 ex.append(u)
 if u==g: break
 for v in neighbors(u):
  nd=du+cost.get(v,1)
  if nd<d.get(v,999): d[v]=nd; par[v]=u; heapq.heappush(h,(nd,v))
p=reconstruct(par,g); draw_grid(p,ex,[x for _,x in h],title=f'UCS cost {d[g]}')
assert d[g]==8 and (2,2) not in p

In [ ]:
from collections import deque
walls={(1,0),(1,1),(1,2),(3,2),(3,3)}; s=(0,0); g=(4,4); q=deque([s]); par={s:None}; ex=[]
while q:
 u=q.popleft(); ex.append(u)
 if u==g: break
 for v in neighbors(u,walls=walls):
  if v not in par: par[v]=u; q.append(v)
p=reconstruct(par,g); draw_grid(p,ex,q,walls=walls,title='BFS frontier around walls')
assert len(p)-1==8 and not any(x in walls for x in p)

In [ ]:
from collections import deque
s=(0,0); g=(4,0)
def run(kind):
 c=deque([s]) if kind=='bfs' else [s]; seen={s}; par={s:None}; ex=[]
 while c:
  u=c.popleft() if kind=='bfs' else c.pop(); ex.append(u)
  if u==g: break
  for v in neighbors(u):
   if v not in seen: seen.add(v); par[v]=u; c.append(v)
 return reconstruct(par,g),ex
bp,be=run('bfs'); dp,de=run('dfs'); draw_grid(bp,be,goal=g,title=f'BFS {len(be)} vs DFS {len(de)} explored')
assert len(bp)-1==4 and len(be)<len(de)